# Latihan Bonus: Server MCP Buatan Sendiri + Klien Pengujinya

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nur-Azzam-Wafiuddin/workshop/blob/uji-colab/colab/hari4/09-mcp-latihan-4.ipynb)

Ini versi **Google Colab**. Isi dan urutan contohnya sama dengan versi VS Code; yang berbeda hanya cell Persiapan di bawah, yang menyalin repository, memasang pustaka, dan mengisi API key di runtime Colab.

1. Pilih **File > Save a copy in Drive** agar perubahan tersimpan di akun Anda sendiri.
2. Jalankan tiga cell **Persiapan Colab** berurutan (sekitar 2-3 menit pada sesi baru).
3. Kerjakan dari atas ke bawah dengan `Shift+Enter`.

Kode jawaban latihan ada di bagian **Kode Jawaban Latihan** pada [PDF bab ini](https://github.com/Nur-Azzam-Wafiuddin/workshop/blob/uji-colab/hari4/09-mcp/09-mcp.pdf). Salin kodenya ke cell yang bertanda `TODO`.

Kalau runtime terputus, file di `/content` (termasuk `.env`) hilang. Jalankan ulang Persiapan, lalu lanjutkan dari cell yang sedang Anda kerjakan.

## Persiapan Colab
Jalankan tiga cell berikut berurutan sebelum cell lain di notebook ini.

In [ ]:
# Persiapan 1 - Menyalin repository dan memasang pustaka (sekitar 2-3 menit pada sesi baru)
import os
import subprocess
import sys

REPO_URL = "https://github.com/Nur-Azzam-Wafiuddin/workshop.git"
REPO = "/content/workshop-genai-migas" if os.path.isdir("/content") else os.path.abspath("workshop-genai-migas")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "-b", "uji-colab", REPO_URL, REPO], check=True)
os.chdir(REPO)

pasang = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "colab/requirements-colab.txt", "-e", "."],
                        capture_output=True, text=True)
if pasang.returncode != 0:
    print(pasang.stderr[-3000:])
    raise RuntimeError("Instalasi pustaka gagal. Jalankan ulang cell ini.")

# "-e ." memasang common dan migas_shared agar subproses (server MCP) dapat mengimpornya; kernel yang sedang berjalan
# belum memuat instalasi baru, jadi folder repo juga ditambahkan ke sys.path
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")
print("Repository dan pustaka siap. Folder kerja:", os.getcwd())

In [ ]:
# Persiapan 2 - Mengisi ID peserta dan API key (dilewati bila .env sudah berisi key)
import re
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv

env = Path(REPO) / ".env"


def rahasia(nama: str):
    """Nilai dari Colab Secrets (ikon kunci di panel kiri), atau None bila tidak ada."""
    try:
        from google.colab import userdata
        return userdata.get(nama)
    except Exception:  # noqa: BLE001 - bukan di Colab, secret belum dibuat, atau akses notebook belum diizinkan
        return None


if not (env.exists() and re.search(r"(?m)^ONLINE_API_KEY=(?!isi-dengan)\S+", env.read_text(encoding="utf-8"))):
    peserta = (rahasia("PARTICIPANT_ID") or input("ID peserta (nomor meja dan nama singkat, contoh P07-andi): ")).strip()
    kunci = (rahasia("ONLINE_API_KEY") or getpass("API key workshop (ketikan tidak ditampilkan): ")).strip()
    if not kunci:
        raise ValueError("API key belum diisi. Jalankan ulang cell ini.")
    isi = (Path(REPO) / ".env.example").read_text(encoding="utf-8")
    for nama, nilai in (("PARTICIPANT_ID", peserta or "P00-nama"), ("ONLINE_API_KEY", kunci)):
        isi = re.sub(rf"(?m)^{nama}=.*$", lambda _, n=nama, v=nilai: f"{n}={v}", isi)
    env.write_text(isi, encoding="utf-8")
    del kunci

# Nilai dari .env dimuat ulang; modul yang sudah diimpor dibuang agar membaca nilai terbaru
load_dotenv(env, override=True)
for modul in [m for m in sys.modules if m.split(".")[0] in ("common", "migas_shared", "rag_pipeline")]:
    del sys.modules[modul]
print("File .env siap untuk peserta", os.environ.get("PARTICIPANT_ID"))

In [ ]:
# Persiapan 3 - Memeriksa koneksi ke model
from migas_shared.llm import chat

try:
    print("Jawaban model:", chat("Balas hanya dengan kata: SIAP").strip())
except Exception as error:  # noqa: BLE001
    raise RuntimeError("Model tidak menjawab. Periksa API key (jalankan ulang Persiapan 2) dan koneksi internet. "
                       f"Pesan asli: {error}") from error

Isi bagian bertanda `TODO(...)` dengan kode dari bagian **Kode Jawaban Latihan** pada PDF bab ini, lalu jalankan sel satu per satu
dengan Shift+Enter. Baris `... = None  # TODO` atau `raise NotImplementedError` berarti kode itu harus Anda ganti.

Server ada di `latihan_3.py`. Skrip ini:

1. Menjalankan server sebagai subproses (stdio) dan mendaftar tool-nya
2. Menguji tool LANGSUNG dengan input yang valid dan tidak valid (tanpa model), diperiksa di kode
3. Menghubungkan agen ke server dan mengajukan satu pertanyaan gabungan

Kode async dijalankan lewat `jalankan(...)` dari `migas_shared.tampil`, yang bekerja di skrip biasa maupun
di Jupyter/Interactive Window.

In [ ]:
# Langkah 1 - Impor dan model
import os
import sys

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.tools import ToolException
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_openai import ChatOpenAI

from migas_shared.config import ROOT
from migas_shared.tampil import jalankan

load_dotenv()

PATH_SERVER = ROOT / "hari4" / "09-mcp" / "latihan_3.py"

model = ChatOpenAI(
    model=os.getenv("ONLINE_MODEL"),
    base_url=os.getenv("ONLINE_BASE_URL"),
    api_key=os.getenv("ONLINE_API_KEY"),
    reasoning_effort=os.getenv("LLM_REASONING_EFFORT", "none"),
)

## Langkah 2 - Mendaftar tool dari server buatan sendiri
Klien membuka sesi per panggilan dan menutupnya sendiri, jadi tidak ada yang perlu ditutup antar sel.

In [ ]:
# TODO(B9.11): buat MultiServerMCPClient untuk server stdio "work_order" yang menjalankan PATH_SERVER dengan sys.executable
client = None  # TODO: ganti dengan implementasi Anda
# TODO(B9.12): ambil tool server dengan jalankan(client.get_tools()), lalu buat kamus tool_per_nama
tools = []  # TODO: ganti dengan implementasi Anda
tool_per_nama = {}  # TODO: ganti dengan implementasi Anda
print(f"Tool dari server buatan sendiri: {list(tool_per_nama)}\n")

## Langkah 3 - Uji langsung 1: daftar work order terbuka
Pengujian tanpa model, diperiksa di kode. Hasil `daftar_wo_terbuka('Kenanga')` harus memuat WO-26-12611.

In [ ]:
print("Uji langsung (tanpa model):")
lolos = 0

# TODO(B9.13): panggil daftar_wo_terbuka('Kenanga') langsung (ainvoke lewat jalankan), lalu ok = hasilnya memuat WO-26-12611; tambahkan ke lolos
ok = False  # TODO: ganti dengan implementasi Anda
lolos += ok
print(f"   {'LOLOS' if ok else 'GAGAL'}  daftar_wo_terbuka('Kenanga') memuat WO-26-12611")

## Langkah 4 - Uji langsung 2: kerugian downtime
12 jam pada 100 bopd harus menghasilkan 50 barel dan USD 3.500.

In [ ]:
hasil_kerugian = str(jalankan(tool_per_nama["hitung_kerugian_downtime"].ainvoke({"downtime_jam": 12, "laju_bopd": 100})))
ok = "50.0 barel" in hasil_kerugian and "3,500" in hasil_kerugian
lolos += ok
print(f"   {'LOLOS' if ok else 'GAGAL'}  hitung_kerugian_downtime(12 jam, 100 bopd) = 50 barel, USD 3.500")

## Langkah 5 - Uji langsung 3: input tidak valid
Area yang tidak dikenal harus ditolak dengan error yang jelas.

In [ ]:
# TODO(B9.14): panggil daftar_wo_terbuka dengan area 'Atlantis'; ok bernilai True hanya bila ToolException muncul
ok = False  # TODO: ganti dengan implementasi Anda
lolos += ok
print(f"   {'LOLOS' if ok else 'GAGAL'}  area tidak dikenal ditolak dengan error")

print(f"\nHasil uji langsung: {lolos}/3 lolos\n")

## Langkah 6 - Agen memakai server
Satu pertanyaan gabungan: daftar work order terbuka dan kerugian downtime.

In [ ]:
# TODO(B9.15): buat agen dari model dan tool server dengan system prompt asisten pemeliharaan (pakai tool, jangan menebak)
agent = None  # TODO: ganti dengan implementasi Anda
pertanyaan = "Work order apa yang terbuka di Kenanga, dan berapa kerugian bila sumurnya mati 48 jam pada laju 90 bopd?"
print(f"Pengguna: {pertanyaan}")
response = jalankan(agent.ainvoke({"messages": [("human", pertanyaan)]}))
panggilan = [c["name"] for m in response["messages"] for c in getattr(m, "tool_calls", [])]
print(f"   Tool yang dipanggil: {panggilan}")
print(f"Agen: {response['messages'][-1].content}")